# 📊 Sales Performance Analysis — A Business Analytics Case Study

## Project Overview
This project analyzes a two-year retail sales dataset (2003–2005) from a wholesale scale-model vehicle distributor to answer a core business question: **where is revenue coming from, and where should management focus attention?**

The analysis moves from descriptive statistics to a customer segmentation model (RFM), translating each finding into a concrete business recommendation.

**Tools:** Python, Pandas, Matplotlib
**Author:** Masoud Kamali


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

pd.set_option("display.max_columns", None)
plt.rcParams["figure.facecolor"] = "white"

# Folder where chart images are saved for the README / report
os.makedirs("../images", exist_ok=True)

print("Welcome to Sales Analysis Project!")

## 1. Load & Inspect the Data

In [ ]:
# The file uses Latin-1 encoding (UTF-8 fails on some special characters)
df = pd.read_csv("../data/sales_data_sample.csv", encoding="latin1")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

### Data Quality Check
A few columns (`ADDRESSLINE2`, `STATE`, `POSTALCODE`, `TERRITORY`) have missing values, but they are not needed for sales analysis. We also check for duplicate rows.

In [ ]:
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
print()
print("Duplicate rows:", df.duplicated().sum())

## 2. Data Cleaning

In [ ]:
# Convert ORDERDATE to a proper datetime type
df["ORDERDATE"] = pd.to_datetime(df["ORDERDATE"])

# Create a Year-Month period column for time-based analysis
df["YearMonth"] = df["ORDERDATE"].dt.to_period("M")

# Drop columns that are not useful for this analysis (contact/address info)
df_clean = df.drop(columns=[
    "ADDRESSLINE1", "ADDRESSLINE2", "PHONE",
    "CONTACTLASTNAME", "CONTACTFIRSTNAME", "POSTALCODE"
])

df_clean.head()

## 3. Overall Sales Distribution

In [ ]:
df_clean["SALES"].describe()

In [ ]:
plt.figure(figsize=(10, 5))
df_clean["SALES"].hist(bins=30, edgecolor="black")
plt.title("Distribution of Sales")
plt.xlabel("Sales ($)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("../images/sales_distribution.png", dpi=150)
plt.show()

### Business Insight
Most individual order lines fall between \$2,000 and \$5,000, with a long tail of larger orders. This suggests the business is driven by a high volume of mid-sized transactions rather than a few very large ones.

## 4. Sales by Country

In [ ]:
country_sales = (
    df_clean.groupby("COUNTRY")["SALES"]
      .sum()
      .sort_values(ascending=False)
)
country_sales

In [ ]:
plt.figure(figsize=(10, 5))
country_sales.plot(kind="bar")
plt.title("Total Sales by Country")
plt.xlabel("Country")
plt.ylabel("Total Sales ($)")
plt.tight_layout()
plt.savefig("../images/sales_by_country.png", dpi=150)
plt.show()

### Business Insight
The United States generated the highest sales among all countries, followed by Spain and France. This indicates that the U.S. market is the company's strongest source of revenue, and expansion or retention efforts in Spain/France could be the next-best opportunity.

## 5. Sales by Product Line

In [ ]:
product_sales = (
    df_clean.groupby("PRODUCTLINE")["SALES"]
      .sum()
      .sort_values(ascending=False)
)
product_sales

In [ ]:
plt.figure(figsize=(10, 5))
product_sales.plot(kind="bar")
plt.title("Total Sales by Product Line")
plt.xlabel("Product Line")
plt.ylabel("Sales ($)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("../images/sales_by_product_line.png", dpi=150)
plt.show()

### Business Insight
Classic Cars generated the highest sales among all product lines, followed by Vintage Cars. Trains had the lowest sales, suggesting lower customer demand for this category and a possible candidate for reduced inventory or a marketing push.

## 6. Monthly Sales Trend

In [ ]:
monthly_sales = df_clean.groupby("YearMonth")["SALES"].sum()
monthly_sales

In [ ]:
plt.figure(figsize=(12, 6))
monthly_sales.plot(marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales ($)")
plt.grid(True)
plt.tight_layout()
plt.savefig("../images/monthly_sales_trend.png", dpi=150)
plt.show()

### Business Insight
Sales peaked consistently during October and November, indicating a clear seasonal trend tied to holiday buying. In contrast, sales remained relatively lower during the first half of each year — inventory and staffing could be planned around this cycle.

## 7. Top 10 Customers

In [ ]:
top_customers = (
    df_clean.groupby("CUSTOMERNAME")["SALES"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)
top_customers

In [ ]:
plt.figure(figsize=(12, 6))
top_customers.plot(kind="bar")
plt.title("Top 10 Customers by Sales")
plt.xlabel("Customer")
plt.ylabel("Total Sales ($)")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y")
plt.tight_layout()
plt.savefig("../images/top_10_customers.png", dpi=150)
plt.show()

### Business Insight
Euro Shopping Channel is the company's most valuable customer, generating a substantially larger share of revenue than any other account. The sizeable gap between the top customer and the rest suggests that maintaining strong relationships with key accounts is essential for sustaining revenue.

## 8. Sales by Deal Size

In [ ]:
deal_sales = (
    df_clean.groupby("DEALSIZE")["SALES"]
      .sum()
      .sort_values(ascending=False)
)
deal_sales

In [ ]:
plt.figure(figsize=(8, 5))
deal_sales.plot(kind="bar")
plt.title("Sales by Deal Size")
plt.xlabel("Deal Size")
plt.ylabel("Total Sales ($)")
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.tight_layout()
plt.savefig("../images/sales_by_deal_size.png", dpi=150)
plt.show()

### Business Insight
Medium-sized deals generated the highest total revenue, followed by small deals. Large deals contributed the least to total sales, indicating that the business relies more on medium-volume transactions than on a small number of large contracts.

## 9. Sales by Order Status

In [ ]:
status_sales = (
    df_clean.groupby("STATUS")["SALES"]
      .sum()
      .sort_values(ascending=False)
)
status_sales

In [ ]:
plt.figure(figsize=(10, 5))
status_sales.plot(kind="bar")
plt.title("Sales by Order Status")
plt.xlabel("Order Status")
plt.ylabel("Total Sales ($)")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y")
plt.tight_layout()
plt.savefig("../images/sales_by_order_status.png", dpi=150)
plt.show()

### Business Insight
Most revenue comes from shipped orders, which indicates successful order fulfillment. Cancelled, disputed, and on-hold orders contribute only a small portion of total revenue, but they are worth monitoring since each one represents lost or delayed cash flow.

## 10. Pricing: How Much Do Orders Discount Off MSRP?

In [ ]:
# PRICEEACH vs MSRP shows how far the actual selling price deviates from the
# manufacturer's suggested price - a simple proxy for discounting behavior.
df_clean["DISCOUNT_PCT"] = (df_clean["MSRP"] - df_clean["PRICEEACH"]) / df_clean["MSRP"] * 100

discount_by_line = (
    df_clean.groupby("PRODUCTLINE")["DISCOUNT_PCT"]
      .mean()
      .sort_values(ascending=False)
)
discount_by_line

In [ ]:
plt.figure(figsize=(10, 5))
discount_by_line.plot(kind="bar", color="darkorange")
plt.title("Average Discount off MSRP by Product Line")
plt.xlabel("Product Line")
plt.ylabel("Average Discount (%)")
plt.xticks(rotation=45, ha="right")
plt.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.savefig("../images/discount_by_product_line.png", dpi=150)
plt.show()

### Business Insight
Some product lines are consistently sold below MSRP while others are sold closer to (or above) list price. Lines with the deepest average discount are the best candidates for a pricing review, since they may be eroding margin more than their sales volume justifies.

## 11. Which Product Line Performs Best in Each Territory?

In [ ]:
territory_product = (
    df_clean.pivot_table(
        index="PRODUCTLINE",
        columns="TERRITORY",
        values="SALES",
        aggfunc="sum"
    )
    .fillna(0)
)
territory_product

In [ ]:
plt.figure(figsize=(10, 6))
plt.imshow(territory_product.values, cmap="YlOrRd", aspect="auto")
plt.colorbar(label="Total Sales ($)")
plt.xticks(range(len(territory_product.columns)), territory_product.columns, rotation=45, ha="right")
plt.yticks(range(len(territory_product.index)), territory_product.index)
plt.title("Sales Heatmap: Product Line vs Territory")
plt.tight_layout()
plt.savefig("../images/territory_product_heatmap.png", dpi=150)
plt.show()

### Business Insight
Classic Cars dominate revenue across almost every territory, confirming it as the company's flagship line globally rather than a regional preference. Territories where a secondary product line stands out are good candidates for localized marketing.

## 12. Correlation Between Numeric Variables

In [ ]:
numeric_cols = ["QUANTITYORDERED", "PRICEEACH", "SALES", "MSRP"]
corr = df_clean[numeric_cols].corr()
corr

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=45, ha="right")
plt.yticks(range(len(numeric_cols)), numeric_cols)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.savefig("../images/correlation_matrix.png", dpi=150)
plt.show()

### Business Insight
`SALES` correlates most strongly with `QUANTITYORDERED`, meaning revenue per order line is driven more by how many units are ordered than by the unit price itself. This suggests volume-based promotions may have more revenue impact than price increases.